# 2 · Reproducibilidad

<a href="https://colab.research.google.com/github/B3njjaman/MLOps-en-Azure/blob/main/modulos/modulo-1/02_reproducibilidad.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"></a>

**Módulo 1 · Del notebook al código reproducible.** La lección 1 terminó con cuatro fallas. Esta resuelve la primera: la promesa del modelo no se repite. La meta es doble: que dos corridas den el mismo modelo, y que de cada modelo se pueda decir con qué se entrenó. Corresponde a la primera condición de la Definición 1.6.

La lección trata el entrenamiento como una función con cuatro entradas: datos, configuración, semillas y entorno. Muestra cómo fijar cada una y cómo dejar constancia de ellas en una *ficha de entrenamiento* que viaja con el modelo. En el camino aparecen dos hechos poco intuitivos: el orden de las filas de la tabla cambia el modelo, y ni siquiera con todo fijo las predicciones son siempre iguales al último decimal. Al final se ve cómo Azure Machine Learning guarda lo mismo: *environments* y *data assets* con versión.

Requisitos: numpy, pandas, scikit-learn y matplotlib, que Colab y Kaggle ya traen. La primera celda descarga el simulador de la planta si no está junto al notebook. Las cifras del texto salen de scikit-learn 1.9 y pandas 3.1. Con otras versiones, como las de Colab, algunas cambian en el último decimal, por lo que explica esta misma lección.

In [1]:
from pathlib import Path
from urllib.request import urlretrieve

if not Path("planta.py").exists():  # en Colab o Kaggle; en el PC ya está junto al notebook
    urlretrieve(
        "https://raw.githubusercontent.com/B3njjaman/MLOps-en-Azure/main/modulos/modulo-1/planta.py",
        "planta.py",
    )

from importlib.metadata import version

for paquete in ("numpy", "pandas", "scikit-learn"):
    print(paquete, version(paquete))

numpy 2.4.6
pandas 3.1.0
scikit-learn 1.9.1


## 1. El entrenamiento como función

En la lección 1, `modelo.fit(X_ent, y_ent)` parecía una instrucción sin más. Para repetirla hay que preguntar de qué depende su resultado.

**Definición 2.1 (entrenamiento).** Un entrenamiento es una función $\mathcal A$ que entrega un modelo $f$ y su ficha:

$$f,\ \text{ficha} = \mathcal A(D,\ \theta,\ s,\ E),$$

donde $D$ son los datos (la tabla y su orden), $\theta$ la configuración (hiperparámetros, columnas usadas, forma de partir los datos), $s$ las semillas de todo lo aleatorio y $E$ el entorno: versiones de Python y de las bibliotecas, y la plataforma. El código de $\mathcal A$ es la función misma; su versión se registra con el commit de Git, tema del módulo 2.

<img src="https://b3njjaman.github.io/MLOps-en-Azure/imagenes/diagramas/modulo-1/entrenamiento-como-funcion.svg" alt="Cuatro cajas a la izquierda: D, datos con su huella SHA-256; theta, configuración en config_entrenamiento.json; s, semillas de la partición y del modelo; E, entorno con requirements.txt y plataforma. Flechas de las cuatro entran a una caja central, A, entrenar(config, df). De ella salen dos flechas a la derecha: el modelo f con la huella de sus predicciones, y la ficha, un JSON con las cuatro huellas. La animación enciende primero los datos, luego la configuración y las semillas, luego el entorno, después la función con sus flechas de entrada, y al final las dos salidas" width="760">

En la lección 1 solo una de las cuatro entradas estaba escrita en alguna parte: los datos venían de `historia()`. La configuración estaba repartida entre celdas, las semillas se agregaron a mitad de la lección y el entorno no se registró. Si alguna de las cuatro cambia sin que nadie lo note, el modelo cambia sin que nadie lo note.

**Definición 2.2 (reproducibilidad).** Sea $\mathcal R$ un conjunto fijo de entradas del modelo, el *conjunto de referencia*. Dos corridas de $\mathcal A$ con las mismas cuatro entradas, que entregan $f$ y $f'$, son

- **exactamente reproducibles** si $f(x) = f'(x)$ para todo $x \in \mathcal R$, bit a bit;
- **numéricamente reproducibles con tolerancia $\varepsilon$** si $\max_{x \in \mathcal R} \lvert f(x) - f'(x) \rvert \le \varepsilon$.

El entrenamiento es reproducible, exacto o numérico, si lo cumple para todo par de corridas con $(D, \theta, s, E)$ iguales.

La distinción importa porque la exacta casi nunca se puede garantizar: las sumas de punto flotante no son asociativas, y si un cálculo se reparte en hilos, el orden de las sumas puede cambiar de una corrida a otra. Como el consumo de la planta se mide con un decimal (0,1 MWh), una tolerancia de $\varepsilon = 10^{-6}$ MWh es más que suficiente. La sección 2 muestra el fenómeno con el bosque de la lección 1.

*Nota.* En GPU el problema es mayor: fijar las semillas no basta, porque algunas operaciones de cuDNN no son deterministas. En PyTorch hay que pedir además `torch.use_deterministic_algorithms(True)`. Este curso usa CPU, y lo que sigue es lo que se necesita ahí.

## 2. Semillas

Las dos fuentes de azar de la lección 1 son `train_test_split`, que elige los días de prueba, y `RandomForestRegressor`, que elige las muestras de cada árbol (*bootstrap*) y el orden en que prueba las features en cada corte. Cada una acepta un `random_state`. Se parte de la lección 1: la misma tabla, las mismas cinco features y la misma partición.

In [2]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split

from planta import historia

df = historia()

X = pd.DataFrame(
    {
        "dia_semana": df["fecha"].dt.day_of_week,
        "produccion": df["produccion"],
        "temperatura": df["temperatura"],
        "consumo_ayer": df["consumo"].shift(1),
        "consumo_semana_pasada": df["consumo"].shift(7),
    }
).iloc[7:]
y = df["consumo"].iloc[7:]
X_ent, X_pru, y_ent, y_pru = train_test_split(X, y, test_size=0.2, random_state=0)


def ajustar(X, y, n_jobs=-1, semilla=0):
    return RandomForestRegressor(n_estimators=200, random_state=semilla, n_jobs=n_jobs).fit(X, y)


# Sin semilla: dos corridas, dos modelos
sin_1 = ajustar(X_ent, y_ent, semilla=None)
sin_2 = ajustar(X_ent, y_ent, semilla=None)
dif = np.abs(sin_1.predict(X_pru) - sin_2.predict(X_pru)).max()
print(f"Sin semilla, diferencia máxima entre dos corridas: {dif:.2f} MWh")

Sin semilla, diferencia máxima entre dos corridas: 2.80 MWh


Sin semilla, las dos corridas dan modelos distintos, y la diferencia (que cambia en cada ejecución) es de varios MWh en el peor día. Con semilla, las dos corridas deberían coincidir. Queda por ver si el reparto del trabajo entre procesadores (`n_jobs`) cambia algo. Se comparan los árboles y las predicciones de tres bosques con la misma semilla:

In [3]:
bosques = {n: ajustar(X_ent, y_ent, n_jobs=n) for n in (1, 2, -1)}

base = bosques[1]
for n, b in bosques.items():
    arboles_iguales = all(
        np.array_equal(a.tree_.threshold, c.tree_.threshold) and np.array_equal(a.tree_.value, c.tree_.value)
        for a, c in zip(base.estimators_, b.estimators_)
    )
    dif = np.abs(base.predict(X_pru) - b.predict(X_pru)).max()
    print(f"n_jobs={n:>2}:  árboles idénticos a n_jobs=1: {arboles_iguales}   diferencia máxima en predicciones: {dif:.1e} MWh")

# El mismo modelo, preguntado 20 veces
repeticiones = [bosques[-1].predict(X_pru) for _ in range(20)]
dif = max(np.abs(r - repeticiones[0]).max() for r in repeticiones)
print(f"Mismo bosque con n_jobs=-1, 20 predicciones seguidas: diferencia máxima {dif:.1e} MWh")

n_jobs= 1:  árboles idénticos a n_jobs=1: True   diferencia máxima en predicciones: 0.0e+00 MWh
n_jobs= 2:  árboles idénticos a n_jobs=1: True   diferencia máxima en predicciones: 5.7e-14 MWh
n_jobs=-1:  árboles idénticos a n_jobs=1: True   diferencia máxima en predicciones: 1.1e-13 MWh


Mismo bosque con n_jobs=-1, 20 predicciones seguidas: diferencia máxima 1.1e-13 MWh


Con `random_state` fijo, `n_jobs` no cambia el bosque: cada árbol recibe su semilla antes de repartir el trabajo, y sus umbrales y valores son idénticos. Lo que sí puede cambiar es la última cifra de la predicción. El bosque promedia sus 200 árboles, y con varios hilos el orden de la suma no está fijo. Las diferencias son de punto flotante, del orden de $10^{-13}$ MWh en la corrida del curso (en otro equipo pueden ser exactamente cero). Es un caso de reproducibilidad numérica sin ser exacta, en el sentido de la Definición 2.2. Por eso la ficha de la sección 6 compara predicciones redondeadas, no los bits.

Hay otra entrada que nadie suele registrar: el **orden de las filas**. El bootstrap de cada árbol elige posiciones al azar, y la posición 17 es la fila que esté en ese lugar. Si se reordena la tabla con las mismas filas, las mismas semillas y los mismos hiperparámetros, el árbol $k$ se entrena con otras filas.

In [4]:
permutacion = np.random.default_rng(1).permutation(len(X_ent))
barajado = ajustar(X_ent.iloc[permutacion], y_ent.iloc[permutacion])

mae_orden = mean_absolute_error(y_pru, bosques[-1].predict(X_pru))
mae_barajado = mean_absolute_error(y_pru, barajado.predict(X_pru))
dif = np.abs(bosques[-1].predict(X_pru) - barajado.predict(X_pru)).max()

print(f"MAE con las filas en orden:    {mae_orden:.2f} MWh")
print(f"MAE con las filas barajadas:   {mae_barajado:.2f} MWh")
print(f"Diferencia máxima entre predicciones: {dif:.2f} MWh")

MAE con las filas en orden:    6.10 MWh
MAE con las filas barajadas:   6.09 MWh
Diferencia máxima entre predicciones: 3.47 MWh


Es el mismo conjunto de filas, y las predicciones cambian hasta en 3,47 MWh en el peor día. El MAE casi no se mueve (6,10 y 6,09), así que la promesa no avisa: es otro modelo con la misma nota.

**Observación 2.1 (el orden es parte de los datos).** Con las mismas filas, la misma semilla y los mismos hiperparámetros, un orden distinto da otro modelo. No es un defecto de scikit-learn: el algoritmo es una función de la secuencia de filas, no del conjunto. Por eso la identidad de $D$ en la Definición 2.1 incluye el orden. Un `ORDER BY` que falta en una consulta a una base de datos, o un `groupby` que devuelve los grupos en otro orden, son formas reales de cambiar el modelo sin cambiar los datos.

## 3. La huella de los datos

Para saber si dos corridas usaron los mismos datos no sirve comparar los nombres de los archivos, ni las fechas de modificación. Sirve una *huella* (*fingerprint*) del contenido.

**Definición 2.3 (huella de los datos).** Sea $\operatorname{ser}(D)$ una serialización canónica de la tabla $D$, es decir, una secuencia de bytes que depende solo del contenido: columnas en orden alfabético, filas en su orden, fechas en formato ISO y números con un número fijo de decimales. La huella de $D$ es

$$h(D) = \operatorname{SHA\text{-}256}\big(\operatorname{ser}(D)\big),$$

un número de 256 bits que se escribe como 64 dígitos hexadecimales.

La serialización fija las decisiones que, si no, harían que la misma tabla diera huellas distintas: el orden de las columnas, el índice, el separador de líneas, el formato de las fechas. Las filas no se ordenan, por la Observación 2.1.

In [5]:
import hashlib


def serializar(tabla):
    # Columnas por nombre, sin índice, fechas ISO, seis decimales, saltos de línea "\n"
    return tabla.to_csv(
        index=False,
        columns=sorted(tabla.columns),
        float_format="%.6f",
        date_format="%Y-%m-%d",
        lineterminator="\n",
    ).encode("utf-8")


def huella_datos(tabla):
    return hashlib.sha256(serializar(tabla)).hexdigest()


print(serializar(df.head(2)).decode())
print("huella:", huella_datos(df))

consumo,fecha,produccion,temperatura,temperatura_pronostico
295.500000,2022-01-01,3040,20.800000,21.200000
248.700000,2022-01-02,2385,21.300000,21.300000

huella: fe1c8d961d934ce9ce8af1decd0518ebdf6f77ae761e9d6c9ca89fe70c96f1b3


Las columnas salieron en orden alfabético y los números con un formato fijo. Lo que se pide de una huella es que dos tablas distintas casi nunca la compartan.

**Propiedad 2.1 (colisiones).** Si se modela SHA-256 como una función que asigna a cada entrada un valor uniforme de 256 bits, la probabilidad de que $n$ entradas distintas tengan alguna colisión es a lo más $n(n-1)/2^{257}$. En particular, con $n = 2^{64}$ tablas es menor que $2^{-128}$.

*Demostración.* Hay $\binom{n}{2} = n(n-1)/2$ pares de entradas, y cada par colisiona con probabilidad $2^{-256}$. Por la cota de la unión, $P(\text{alguna colisión}) \le \frac{n(n-1)}{2}\,2^{-256} = n(n-1)\,2^{-257}$. Con $n = 2^{64}$, $n(n-1) < 2^{128}$ y la cota es menor que $2^{-129}$. $\square$

El supuesto es una idealización: nadie ha publicado una colisión de SHA-256, pero eso no es una prueba. Aquí la huella sirve para identificar, no para defenderse de alguien que fabrique tablas a propósito, y para eso la cota basta con holgura.

Una huella sirve si cambia cuando debe y no cambia cuando no debe. Se prueban tres modificaciones de la tabla de la planta:

In [6]:
h0 = huella_datos(df)

corregido = df.copy()
corregido.loc[100, "consumo"] += 0.1  # se corrige un solo valor, en una décima

casos = {
    "tabla original": df,
    "se corrige un valor de consumo en 0,1": corregido,
    "se reordenan las columnas": df[df.columns[::-1]],
    "se reordenan las filas (se invierte la tabla)": df.iloc[::-1],
}
for nombre, tabla in casos.items():
    h = huella_datos(tabla)
    print(f"{h[:16]}  {'igual' if h == h0 else 'distinta':>8}  {nombre}")

fe1c8d961d934ce9     igual  tabla original
2191f58e52e4d6e8  distinta  se corrige un valor de consumo en 0,1
fe1c8d961d934ce9     igual  se reordenan las columnas
d3fa7963b28bea2a  distinta  se reordenan las filas (se invierte la tabla)


Un solo valor, cambiado en una décima de MWh, cambia la huella completa. Reordenar las columnas no la cambia, porque la serialización las ordena por nombre. Reordenar las filas sí la cambia, y es lo correcto: por la Observación 2.1 es otro modelo.

Con la huella en la mano se puede decir «este modelo se entrenó con la tabla `a1b2…`» y comprobarlo después. El costo es una pasada sobre la tabla.

Una limitación: con `float_format="%.6f"`, dos tablas que difieren en menos de $10^{-6}$ en un valor tienen la misma huella. Para estos datos, que tienen un decimal, sobra. Para otros hay que elegir el formato con criterio y dejarlo escrito.

## 4. La configuración, fuera del código

Los hiperparámetros, las columnas y la partición estaban escritos a mano en celdas. Si cambian, nadie queda enterado. La alternativa es un archivo que la función de entrenamiento lee, y que se versiona junto al código.

In [7]:
import json

config = {
    "features": ["dia_semana", "produccion", "temperatura", "consumo_ayer", "consumo_semana_pasada"],
    "objetivo": "consumo",
    "modelo": {"tipo": "RandomForestRegressor", "n_estimators": 200},
    "particion": {"tipo": "aleatoria", "prueba": 0.2},
    "semillas": {"particion": 0, "modelo": 0},
}
SALIDAS = Path("salidas")  # lo que escribe la lección queda aquí
SALIDAS.mkdir(exist_ok=True)
(SALIDAS / "config_entrenamiento.json").write_text(json.dumps(config, indent=2, ensure_ascii=False))

config = json.loads((SALIDAS / "config_entrenamiento.json").read_text())
print((SALIDAS / "config_entrenamiento.json").read_text())

{
  "features": [
    "dia_semana",
    "produccion",
    "temperatura",
    "consumo_ayer",
    "consumo_semana_pasada"
  ],
  "objetivo": "consumo",
  "modelo": {
    "tipo": "RandomForestRegressor",
    "n_estimators": 200
  },
  "particion": {
    "tipo": "aleatoria",
    "prueba": 0.2
  },
  "semillas": {
    "particion": 0,
    "modelo": 0
  }
}


El archivo reúne los argumentos $\theta$ y $s$ de la Definición 2.1. `n_jobs` no aparece: por la sección 2 no cambia el bosque, y es una decisión de la máquina, no del experimento. En el módulo 3 este mismo archivo pasa a ser un YAML que el pipeline recibe como parámetro.

## 5. El entorno

La Definición 2.1 incluye $E$. Una actualización de scikit-learn puede cambiar cómo se eligen los cortes de un árbol, y entonces los mismos datos, la misma configuración y las mismas semillas dan otro modelo. Lo mínimo es registrar las versiones y, para repetir la corrida, poder reinstalarlas.

In [8]:
import platform

PAQUETES = ("numpy", "pandas", "scikit-learn")


def entorno():
    return {
        "python": platform.python_version(),
        "plataforma": platform.platform(),
        "paquetes": {p: version(p) for p in PAQUETES},
    }


requisitos = "\n".join(f"{p}=={version(p)}" for p in PAQUETES) + "\n"
(SALIDAS / "requirements.txt").write_text(requisitos)

print(json.dumps(entorno(), indent=2))
print("--- requirements.txt")
print(requisitos)

{
  "python": "3.11.8",
  "plataforma": "Windows-10-10.0.19045-SP0",
  "paquetes": {
    "numpy": "2.4.6",
    "pandas": "3.1.0",
    "scikit-learn": "1.9.1"
  }
}
--- requirements.txt
numpy==2.4.6
pandas==3.1.0
scikit-learn==1.9.1



`pip install -r requirements.txt` en un entorno limpio reinstala exactamente esas versiones. Pide `==`, no `>=`: con `>=` el entorno de hoy y el de dentro de seis meses pueden ser distintos. Quedan fuera de este archivo la versión de Python y la plataforma. La primera se fija en el `conda.yaml` o en la imagen de Docker (sección 7), y la segunda se registra para saber en qué equipo se entrenó, porque las bibliotecas de álgebra lineal pueden dar sumas distintas en procesadores distintos.

## 6. La ficha de entrenamiento

Con las cuatro entradas identificadas, falta juntarlas en una sola función que devuelva el modelo y la constancia de cómo se hizo.

**Definición 2.4 (ficha de entrenamiento).** La ficha es un documento JSON con: la huella $h(D)$ de los datos, la configuración $\theta$, las semillas $s$, el entorno $E$, las métricas, la fecha y una huella del modelo.

La huella del modelo plantea una pregunta: ¿sobre qué se calcula? La respuesta obvia es el archivo del modelo, el `pickle`. No sirve, y se puede ver con los bosques de la sección 2: los de `n_jobs=1` y `n_jobs=-1` tienen árboles idénticos y la misma función, pero su serialización incluye el parámetro `n_jobs`.

In [9]:
import pickle

pickle_1 = hashlib.sha256(pickle.dumps(bosques[1])).hexdigest()
pickle_menos1 = hashlib.sha256(pickle.dumps(bosques[-1])).hexdigest()
print(f"huella del pickle, n_jobs=1:   {pickle_1[:16]}")
print(f"huella del pickle, n_jobs=-1:  {pickle_menos1[:16]}")
print(f"¿iguales? {pickle_1 == pickle_menos1}")

huella del pickle, n_jobs=1:   9dc70292c5914a32
huella del pickle, n_jobs=-1:  a7a75bd47e121642
¿iguales? False


El `pickle` guarda también detalles que no son parte de lo que el modelo hace: parámetros de ejecución como `n_jobs` y detalles de la versión de la biblioteca. Hay dos corridas igual de reproducibles con `pickle` distinto. Lo que importa es la función $f$, y una función se identifica por lo que devuelve. La huella del modelo se calcula entonces sobre sus **predicciones en un conjunto fijo $\mathcal R$**, redondeadas a seis decimales, como pide la Definición 2.2 con $\varepsilon \approx 10^{-6}$.

El conjunto de referencia es una malla escrita en el código: los siete días de la semana, diez temperaturas entre 4 y 26 °C, y producción y consumos típicos. No depende de los datos de entrenamiento, así que dos modelos entrenados con datos distintos se comparan sobre las mismas entradas.

El redondeo tiene un borde. Una diferencia de $10^{-13}$ cambia el sexto decimal solo si el valor cae a menos de $10^{-13}$ del punto medio entre dos múltiplos de $10^{-6}$. Para un valor, eso ocurre con probabilidad del orden de $10^{-7}$, y para los 70 de la malla, del orden de $10^{-5}$. Es raro, pero no imposible, y si ocurriera, la comparación de fichas fallaría sin que el modelo sea otro. Por eso el criterio final de reproducibilidad numérica es la diferencia máxima $\le \varepsilon$, y la huella es un atajo. Hay un segundo límite: la malla deja fijos los dos consumos en 280 MWh, así que dos bosques que solo difieran con otros consumos darían la misma huella. Con 200 árboles es muy improbable, pero la huella resume al modelo y no lo prueba entero.

In [10]:
import itertools
from datetime import datetime

REFERENCIA = pd.DataFrame(
    [
        {
            "dia_semana": dia,
            "produccion": 2300.0 if dia == 6 else 3000.0,
            "temperatura": float(temp),
            "consumo_ayer": 280.0,
            "consumo_semana_pasada": 280.0,
        }
        for dia, temp in itertools.product(range(7), np.linspace(4, 26, 10).round(1))
    ]
)


def huella_modelo(modelo, features):
    pred = modelo.predict(REFERENCIA[features])
    texto = "\n".join(f"{v:.6f}" for v in pred)
    return hashlib.sha256(texto.encode()).hexdigest()


def armar_features(tabla, features):
    X = pd.DataFrame(
        {
            "dia_semana": tabla["fecha"].dt.day_of_week,
            "produccion": tabla["produccion"],
            "temperatura": tabla["temperatura"],
            "consumo_ayer": tabla["consumo"].shift(1),
            "consumo_semana_pasada": tabla["consumo"].shift(7),
        }
    )
    return X[features].iloc[7:], tabla["consumo"].iloc[7:]


def entrenar(config, tabla):
    # Esta función sabe entrenar una sola combinación; si la configuración pide otra, falla en vez de mentir en la ficha.
    assert config["objetivo"] == "consumo"
    assert config["modelo"]["tipo"] == "RandomForestRegressor"
    assert config["particion"]["tipo"] == "aleatoria"
    features = config["features"]
    semillas = config["semillas"]
    X, y = armar_features(tabla, features)
    X_ent, X_pru, y_ent, y_pru = train_test_split(
        X, y, test_size=config["particion"]["prueba"], random_state=semillas["particion"]
    )
    modelo = RandomForestRegressor(
        n_estimators=config["modelo"]["n_estimators"], random_state=semillas["modelo"], n_jobs=-1
    ).fit(X_ent, y_ent)

    ficha = {
        "fecha": datetime.now().isoformat(timespec="seconds"),
        "huella_datos": huella_datos(tabla),
        "config": config,
        "semillas": semillas,
        "entorno": entorno(),
        "metricas": {"mae_prueba": round(mean_absolute_error(y_pru, modelo.predict(X_pru)), 6)},
        "huella_modelo": huella_modelo(modelo, features),
    }
    return modelo, ficha

La función no tiene estado oculto: todo lo que usa entra por `config` y `tabla`, o es el entorno, y todo lo que deja constancia sale en la ficha. Se prueba la propiedad que importa, con dos llamadas:

In [11]:
modelo_a, ficha_a = entrenar(config, df)
modelo_b, ficha_b = entrenar(config, df)


def sin_fecha(ficha):
    return json.dumps({k: v for k, v in ficha.items() if k != "fecha"}, sort_keys=True)


print("Fichas idénticas salvo la fecha:", sin_fecha(ficha_a) == sin_fecha(ficha_b))
dif = np.abs(modelo_a.predict(X_pru) - modelo_b.predict(X_pru)).max()
print(f"Diferencia máxima entre predicciones: {dif:.1e} MWh")

(SALIDAS / "ficha_entrenamiento.json").write_text(json.dumps(ficha_a, indent=2, ensure_ascii=False))
print(json.dumps(ficha_a, indent=2, ensure_ascii=False))

Fichas idénticas salvo la fecha: True
Diferencia máxima entre predicciones: 1.1e-13 MWh
{
  "fecha": "2026-10-29T20:18:03",
  "huella_datos": "fe1c8d961d934ce9ce8af1decd0518ebdf6f77ae761e9d6c9ca89fe70c96f1b3",
  "config": {
    "features": [
      "dia_semana",
      "produccion",
      "temperatura",
      "consumo_ayer",
      "consumo_semana_pasada"
    ],
    "objetivo": "consumo",
    "modelo": {
      "tipo": "RandomForestRegressor",
      "n_estimators": 200
    },
    "particion": {
      "tipo": "aleatoria",
      "prueba": 0.2
    },
    "semillas": {
      "particion": 0,
      "modelo": 0
    }
  },
  "semillas": {
    "particion": 0,
    "modelo": 0
  },
  "entorno": {
    "python": "3.11.8",
    "plataforma": "Windows-10-10.0.19045-SP0",
    "paquetes": {
      "numpy": "2.4.6",
      "pandas": "3.1.0",
      "scikit-learn": "1.9.1"
    }
  },
  "metricas": {
    "mae_prueba": 6.1015
  },
  "huella_modelo": "b85d33654d7ae2864c79b008dd8a3ee2ea87309aae698df6223d6eb2d7fafa7b

**Propiedad 2.2 (la ficha detecta).** Si $\mathcal A$ es reproducible en el sentido de la Definición 2.2, dos corridas con las mismas cuatro entradas dan fichas iguales, salvo la fecha y los bordes de redondeo descritos arriba. Dicho en contrapositiva: si dos corridas que debían tener las mismas entradas dan fichas distintas, alguna entrada no estaba realmente fija.

*Demostración.* Cada campo de la ficha es función de las cuatro entradas: $h(D)$ y $E$ dependen de $D$ y $E$, la configuración y las semillas son $\theta$ y $s$, y la huella del modelo y las métricas dependen de $f$, que por reproducibilidad es la misma salvo $\varepsilon$. La segunda afirmación es la contrapositiva de la primera. $\square$

La contrapositiva es lo útil: cuando dos fichas que deberían coincidir difieren, los campos `huella_datos`, `config` y `entorno` dicen qué entrada cambió. La huella del modelo y las métricas se mueven con cualquier cambio, así que solo confirman que el modelo es otro. Para comprobarlo, se cambia una entrada a la vez y se mira qué campos se mueven:

In [12]:
def diferencias(a, b):
    return [k for k in a if k != "fecha" and json.dumps(a[k], sort_keys=True) != json.dumps(b[k], sort_keys=True)]


cambios = {
    "un valor de consumo corregido": (config, corregido),
    "n_estimators: 200 a 201": ({**config, "modelo": {**config["modelo"], "n_estimators": 201}}, df),
    "semilla de la partición: 0 a 1": ({**config, "semillas": {**config["semillas"], "particion": 1}}, df),
    "semilla del modelo: 0 a 1": ({**config, "semillas": {**config["semillas"], "modelo": 1}}, df),
}
for nombre, (cfg, tabla) in cambios.items():
    _, ficha = entrenar(cfg, tabla)
    print(f"{nombre}:\n    cambian {diferencias(ficha_a, ficha)}")

un valor de consumo corregido:
    cambian ['huella_datos', 'metricas', 'huella_modelo']


n_estimators: 200 a 201:
    cambian ['config', 'metricas', 'huella_modelo']


semilla de la partición: 0 a 1:
    cambian ['config', 'semillas', 'metricas', 'huella_modelo']


semilla del modelo: 0 a 1:
    cambian ['config', 'semillas', 'metricas', 'huella_modelo']


Cada cambio deja su marca en los campos que le corresponden. Con la ficha guardada junto al modelo, la pregunta «¿con qué se entrenó este?» tiene respuesta, y la pregunta «¿es el mismo que el de ayer?» se resuelve comparando dos JSON.

La ficha no resuelve todo. No dice si el modelo es *bueno*, solo que se puede repetir. El `mae_prueba` de la ficha es el 6,10 de la lección 1, y sigue dependiendo de la partición: los ejercicios muestran que reproducible no es lo mismo que fiable.

## 7. En Azure

La Definición 2.1 tiene un equivalente en Azure Machine Learning para cada entrada.

| Entrada | En Azure Machine Learning |
|---|---|
| $D$, datos | *Data asset*: una referencia versionada e inmutable a los datos (`uri_file`, `uri_folder` o `mltable`) |
| $\theta$ y $s$, configuración y semillas | Un archivo en el repositorio, o parámetros del job; el job los registra |
| $E$, entorno | *Environment*: una imagen de Docker más, opcionalmente, un `conda.yaml`. Una versión es inmutable |
| La ficha | Las métricas, los parámetros y los artefactos del job, con MLflow |

Un *environment* definido por uno mismo es lo que se usa en producción. Se construye una vez y queda en el Azure Container Registry del workspace. Los de tipo *curated* son de Microsoft, inmutables y sirven para empezar. Como una versión de un environment no cambia, actualizar una dependencia crea una versión nueva, que es lo que se quiere: queda registrado qué versión usó cada modelo.

El `conda.yaml` es el `requirements.txt` de la sección 5 con la versión de Python:

```yaml
# conda.yaml
name: forecast-env
channels:
  - conda-forge
dependencies:
  - python=3.11
  - pip
  - pip:
      - numpy==2.4.6        # las versiones que imprimió la sección 5
      - pandas==3.1.0
      - scikit-learn==1.9.1
```

```yaml
# environment.yml
$schema: https://azuremlschemas.azureedge.net/latest/environment.schema.json
name: forecast-env
version: 1
image: mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu22.04:20261026.v1   # etiqueta fija: latest cambia cada semana
conda_file: conda.yaml
```

```yaml
# data.yml
$schema: https://azuremlschemas.azureedge.net/latest/data.schema.json
name: consumo-diario
version: 1
type: uri_file
path: ./consumo_diario.csv
description: Consumo de la planta, 2022 a 2025
```

Ambos se registran con la CLI v2: `az ml environment create --file environment.yml` y `az ml data create --file data.yml`. Los comandos del curso suponen que el grupo de recursos y el workspace se fijaron una vez con `az configure --defaults group=<grupo> workspace=<workspace>`; si no, cada comando lleva `--resource-group` y `--workspace-name`. La versión de un data asset es de quien la escribe: si se corrige un valor, el contenido nuevo debe registrarse como la versión 2, y la huella $h(D)$ de la sección 3 sirve para comprobar que el archivo de la versión 1 sigue siendo el que era.

La imagen también lleva su versión: Microsoft publica una etiqueta nueva de la imagen base casi cada semana, y `latest` apunta siempre a la última, así que un environment construido con `latest` en dos fechas puede traer bibliotecas del sistema distintas.

Un *command job* une las piezas. Sus referencias llevan la versión escrita, y eso es lo que importa en esta lección:

```yaml
# job.yml
$schema: https://azuremlschemas.azureedge.net/latest/commandJob.schema.json
code: src
command: >-
  python train.py --data ${{inputs.consumo}} --config config_entrenamiento.json
inputs:
  consumo:
    type: uri_file
    path: azureml:consumo-diario:1
environment: azureml:forecast-env:1
compute: azureml:cpu-cluster
experiment_name: consumo-energia
display_name: entrenamiento-reproducible
```

El job corre dentro de la copia de `src/` que sube Azure ML, así que `config_entrenamiento.json` tiene que estar en esa carpeta, versionado junto a `train.py`.

Azure ML también acepta referencias que terminan en `@latest`, como `azureml:consumo-diario@latest`. Para explorar es cómodo, pero no es reproducible: la misma línea apunta a datos o a un environment distintos según el día. Con la versión escrita, el job registra qué versión de cada data asset usó, y eso queda en el **linaje** del modelo: de un modelo registrado se puede subir hasta el job que lo entrenó, y de ahí a la versión de los datos y al environment. Dentro de `train.py`, la ficha de la sección 6 se guarda con la API de MLflow (`mlflow.log_artifact("ficha_entrenamiento.json")`; en un job de Azure ML, MLflow ya viene conectado al workspace) junto a las métricas y los parámetros.

La ficha y el linaje de Azure se complementan. El linaje dice qué versiones usó el job. La ficha dice además la huella de los datos, que no depende de ningún nombre, y la huella de las predicciones del modelo, con la que se comprueba que reentrenar da lo mismo.

## Ejercicios

**Ejercicio 1 (qué cambia la huella).** Para cada cambio de la lista, decir si la huella $h(D)$ de la sección 3 cambia, y si *debería* cambiar. Después comprobarlo.

1. Pasar `produccion` de entero a decimal (`astype(float)`).
2. Renombrar la columna `temperatura` a `temp`.
3. Agregar una columna que el modelo no usa.
4. Convertir `fecha` a texto (`"2022-01-01"`).
5. Sumar $10^{-9}$ a la columna `temperatura`.

*Pista:* mirar `serializar(df.head(2))` de cada versión.

<details>
<summary>Solución</summary>

```python
h0 = huella_datos(df)
casos = {
    "produccion a decimal": df.assign(produccion=df["produccion"].astype(float)),
    "temperatura renombrada": df.rename(columns={"temperatura": "temp"}),
    "columna nueva": df.assign(extra=0),
    "fecha como texto": df.assign(fecha=df["fecha"].dt.strftime("%Y-%m-%d")),
    "temperatura + 1e-9": df.assign(temperatura=df["temperatura"] + 1e-9),
}
for nombre, tabla in casos.items():
    print(f"{'igual' if huella_datos(tabla) == h0 else 'distinta':>8}  {nombre}")
```

| cambio | ¿cambia la huella? |
|---|---|
| `produccion` a decimal | sí |
| `temperatura` renombrada | sí |
| columna nueva | sí |
| `fecha` como texto | no |
| `temperatura` + 1e-9 | no |


La huella mira la tabla serializada, no lo que el modelo usa ni lo que las personas consideran «el mismo dato». Que pasar de entero a decimal cambie la huella es incómodo: los valores son los mismos. La salida correcta no es hacer la huella más tolerante, sino fijar los tipos de cada columna con una validación, que es lo que hará la lección 5. Una columna nueva sí debe cambiarla, porque es otra tabla, aunque el modelo no la use. Renombrar una columna también: el nombre es parte del contrato. Convertir la fecha a texto no la cambia, porque la serialización escribe las fechas en el mismo formato, y es lo que se quiere. Los $10^{-9}$ quedan por debajo de la resolución de seis decimales y tampoco se notan.

</details>

**Ejercicio 2 (float32 contra float64).** `RandomForestRegressor` convierte las features a float32 antes de entrenar. Entrenar el bosque con las features como float32 y como float64, y después con el objetivo `y` como float32. Medir en cada caso la diferencia máxima entre predicciones sobre `X_pru`.

*Pista:* un número como 281,3 no se representa exactamente en float32.

<details>
<summary>Solución</summary>

```python
ref = ajustar(X_ent, y_ent, n_jobs=1)
p_ref = ref.predict(X_pru)

x32 = ajustar(X_ent.astype("float32"), y_ent, n_jobs=1)
y32 = ajustar(X_ent, y_ent.astype("float32"), n_jobs=1)
print(f"features en float32:  {np.abs(p_ref - x32.predict(X_pru.astype('float32'))).max():.2e} MWh")
print(f"objetivo en float32:  {np.abs(p_ref - y32.predict(X_pru)).max():.2e} MWh")
```

| caso | diferencia máxima |
|---|---|
| features en float32 | 0 |
| objetivo en float32 | cerca de 0,7 MWh (cambia con la versión de scikit-learn) |


Las features dan diferencia cero, porque scikit-learn las trabaja en float32 de todos modos. El objetivo en float32 cambia las sumas de cuadrados con que se eligen los cortes, y algún corte cambia: el modelo es otro, y la diferencia es mucho mayor que cualquier $\varepsilon$ razonable. Es un cambio de $D$ que una huella calculada antes de convertir el tipo no detecta. Conviene calcularla sobre la tabla tal como entra a `fit`.

</details>

**Ejercicio 3 (la predicción no depende de `n_jobs`).** Argumentar por qué, con `random_state` fijo, los árboles no dependen de `n_jobs`, y por qué las predicciones pueden diferir en la última cifra. Comprobar con `n_jobs` en $\{1, 2, 4, -1\}$.

*Pista:* ¿quién decide la semilla de cada árbol, y en qué orden se suman las predicciones de los árboles?

<details>
<summary>Solución</summary>

El bosque saca de `random_state` una semilla por árbol antes de repartir el trabajo, y cada árbol es función de su semilla y de los datos, así que no importa en qué hilo se ajuste. La predicción es $\frac{1}{200}\sum_k T_k(x)$. Cada hilo suma lo suyo en un acumulador común, y el orden en que llegan los hilos no está fijado. Como la suma de números de punto flotante no es asociativa, el resultado puede cambiar en la última cifra, y de ahí la reproducibilidad numérica y no exacta.

```python
base = ajustar(X_ent, y_ent, n_jobs=1)
for n in (1, 2, 4, -1):
    b = ajustar(X_ent, y_ent, n_jobs=n)
    iguales = all(np.array_equal(p.tree_.value, q.tree_.value) for p, q in zip(base.estimators_, b.estimators_))
    print(f"n_jobs={n:>2}: árboles idénticos {iguales},  diferencia máxima {np.abs(base.predict(X_pru) - b.predict(X_pru)).max():.1e}")
```

Los árboles son idénticos en los cuatro casos. Las predicciones difieren a lo más en el orden de $10^{-13}$ MWh con varios hilos, y en 0 con `n_jobs=1` (en otro equipo, las diferencias pueden ser otras, o ninguna).

</details>

**Ejercicio 4 (reproducible no es fiable).** Con `entrenar`, cambiar solo la semilla de la partición de 0 a 5 y mirar el `mae_prueba` de cada ficha. Cada ficha es reproducible. ¿Qué dice esto sobre la promesa del modelo?

*Pista:* es el mismo experimento de la sección 5.1 de la lección 1.

<details>
<summary>Solución</summary>

```python
for semilla in range(6):
    cfg = {**config, "semillas": {**config["semillas"], "particion": semilla}}
    _, ficha = entrenar(cfg, df)
    print(f"partición {semilla}:  MAE {ficha['metricas']['mae_prueba']:.2f}")
```

| partición | MAE |
|---|---|
| 0 | 6,10 |
| 1 | 5,96 |
| 2 | 6,05 |
| 3 | 6,84 |
| 4 | 6,59 |
| 5 | 6,06 |


Cada corrida se puede repetir y cada ficha dice con qué semilla salió, pero el número que prometen va de 5,96 a 6,84 según la partición, que es un azar de otro tipo: aquí es el azar de qué días tocaron en prueba. Fijar la semilla vuelve la promesa repetible, no la vuelve cierta. Un modelo cuya promesa depende de una semilla elegida a mano no tiene una promesa, tiene una entre varias. La **lección 3** quita ese azar con una partición temporal, que no tiene semilla: entrena con el pasado y prueba con lo que viene después.

</details>

**Siguiente:** la lección 3 cambia la forma de evaluar: una validación temporal con backtesting de origen móvil, para que la promesa del modelo se parezca al error que de verdad tendrá.